# CNT retarded polarization: GW/NEGF vs equilibrium RPA

Goal: compare the frequency dependence of the retarded polarization trace obtained from the GW/NEGF workflow with the retarded polarization trace obtained directly from the equilibrium RPA workflow.

Both workflows use the same CNT input files: `inputs/hamiltonian.mat` and `inputs/coulomb_matrix.mat`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from quatrex.core.config import parse_config

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (8.5, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.frameon": False,
})

ROOT = Path.cwd()
RPA_CONFIG = ROOT / "quatrex_config_rpa_smooth.toml"
RPA_RAW = ROOT / "outputs_RPA_smooth" / "raw_rpa_debug"
GW_OUT = ROOT / "outputs_equilibrium_validation"

config = parse_config(RPA_CONFIG)
transport_energies = np.linspace(
    config.electron.energy_window_min,
    config.electron.energy_window_max,
    config.electron.energy_window_num,
)
response_energies_eV = transport_energies - transport_energies[0] + 1e-6
hbar_eV_s = 6.582119569e-16
omega_rad_s = response_energies_eV / hbar_eV_s

def load(path):
    return np.load(path, mmap_mode="r")

def plot_complex_trace(ax, x, y, label, *, color=None, linestyle="-"):
    line, = ax.plot(x, np.real(y), label=f"Re {label}", color=color, linestyle=linestyle, linewidth=1.8)
    imag_color = color if color is not None else line.get_color()
    ax.plot(x, np.imag(y), label=f"Im {label}", color=imag_color, linestyle="--", linewidth=1.8)

print(f"RPA config: {RPA_CONFIG.name}")
print(f"Raw RPA folder: {RPA_RAW}")
print(f"GW/NEGF folder: {GW_OUT}")

## 1. GW/NEGF retarded polarization trace

This is the trace of the retarded GW/NEGF polarization diagonal written by the equilibrium one-iteration validation run. This is the closest GW/NEGF analogue of the raw RPA matrix trace plotted below.


In [ ]:
p_gw_diagonal = np.asarray(load(GW_OUT / "p_retarded_diagonal_0.npy"))
p_gw_trace = p_gw_diagonal.sum(axis=-1)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_complex_trace(
    ax,
    omega_rad_s,
    p_gw_trace,
    "$P^R_{GW}(\omega)$",
)
ax.set_title("GW/NEGF retarded polarization trace")
ax.set_xlabel("angular frequency $\omega$ [rad/s]")
ax.set_ylabel("trace $P^R_{GW}$")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.show()


## 2. q-averaged raw equilibrium RPA retarded polarization trace

This is obtained directly from the equilibrium RPA export. The export builds Bloch bands from `hamiltonian.mat`, evaluates the retarded RPA/Lindhard response, and writes `P_RPA^R(q, omega)`. Following Anders's suggestion, we trace over the orbital matrix for each `q` and then average over all sampled `q` points, leaving one real line and one imaginary line versus angular frequency.

The absolute scale is not forced to match the GW/NEGF trace here; the goal is to compare spectral structure and identify the remaining normalization factor.


In [ ]:
rpa_freqs_eV = np.asarray(load(RPA_RAW / "frequencies_eV.npy"))
rpa_omega_rad_s = rpa_freqs_eV / hbar_eV_s
q_points = np.asarray(load(RPA_RAW / "q_points.npy"))
p_rpa = load(RPA_RAW / "polarization_retarded_qw.npy")

p_rpa_trace_qw = np.trace(p_rpa, axis1=2, axis2=3)
p_rpa_trace_q_mean = np.asarray(p_rpa_trace_qw).mean(axis=0)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(
    rpa_omega_rad_s,
    p_rpa_trace_q_mean.real,
    color="tab:blue",
    linestyle="-",
    linewidth=1.8,
    label=f"Re mean over {len(q_points)} q-points",
)
ax.plot(
    rpa_omega_rad_s,
    p_rpa_trace_q_mean.imag,
    color="tab:blue",
    linestyle="--",
    linewidth=1.8,
    label=f"Im mean over {len(q_points)} q-points",
)

ax.set_title("q-averaged raw RPA $P^R_{RPA}(q, \omega)$ trace")
ax.set_xlabel("angular frequency $\omega$ [rad/s]")
ax.set_ylabel("mean$_q$ trace $P^R_{RPA}$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.show()


## 3. Per-unit-cell normalization diagnostic

The two traces above are not traces over the same size object:

- GW/NEGF trace = trace over the full finite transport device.
- q-averaged RPA trace = mean over `q` of traces over one Bloch unit-cell polarization matrix.

The GW/NEGF device is still built from unit cells (`construct_from_unit_cell = true`), but its diagonal output contains all device-basis entries. The diagnostic below divides the GW/NEGF trace by the ratio between the GW diagonal length and the RPA unit-cell orbital count. This tests whether a per-unit-cell normalization brings the amplitude closer to the q-averaged raw RPA trace.


In [ ]:
gw_basis_size = p_gw_diagonal.shape[-1]
rpa_unit_cell_orbitals = p_rpa.shape[-1]
num_transport_cells = config.device.num_transport_cells
basis_ratio = gw_basis_size / rpa_unit_cell_orbitals
p_gw_trace_per_unit_cell = p_gw_trace / basis_ratio

print(f"GW diagonal length: {gw_basis_size}")
print(f"RPA unit-cell orbital count: {rpa_unit_cell_orbitals}")
print(f"Configured transport cells: {num_transport_cells}")
print(f"GW/RPA basis ratio: {basis_ratio:g}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_complex_trace(
    ax,
    omega_rad_s,
    p_gw_trace_per_unit_cell,
    f"$P^R_{{GW}}(\omega)$ / {basis_ratio:g}",
    color="black",
)
ax.plot(
    rpa_omega_rad_s,
    p_rpa_trace_q_mean.real,
    color="tab:blue",
    linestyle="-",
    label="Re mean$_q$ RPA",
)
ax.plot(
    rpa_omega_rad_s,
    p_rpa_trace_q_mean.imag,
    color="tab:blue",
    linestyle="--",
    label="Im mean$_q$ RPA",
)

ax.set_title("Per-unit-cell GW/NEGF trace vs q-averaged raw RPA trace")
ax.set_xlabel("angular frequency $\omega$ [rad/s]")
ax.set_ylabel("polarization trace")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
plt.show()
